# 🌾 All-In-One Crop Disease Classifier Training Pipeline (Kaggle GPU)
This single notebook downloads the 14 Kaggle datasets directly, indexes them, trains MobileNetV2, and exports `crop_model.onnx`.

**Instructions:** Make sure Accelerator is set to **GPU T4** and click **Run All**!

In [ ]:
# ==============================================================================
# 🌾 ALL-IN-ONE CROP MODEL TRAINER FOR KAGGLE GPU
# Copy and paste this ENTIRE block into a single code cell on Kaggle and run it!
# ==============================================================================

!pip install -q tf2onnx onnx onnxruntime kagglehub

import os
import re
import json
import shutil
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import tf2onnx
import onnxruntime as ort
import kagglehub

print("1. TensorFlow Version:", tf.__version__)
print("   GPU Status:", tf.config.list_physical_devices('GPU'))

# ------------------------------------------------------------------------------
# 1. DOWNLOAD THE 14 DATASETS DIRECTLY VIA KAGGLEHUB (Takes ~1 min in cloud)
# ------------------------------------------------------------------------------
DATASET_SLUGS = [
    ("Rice", "vbookshelf/rice-leaf-diseases"),
    ("Wheat", "kushagra3204/wheat-plant-diseases"),
    ("Potato", "mgmitesh/potato-disease-detection-dataset"),
    ("Tomato", "kaustubhb999/tomatoleaf"),
    ("Onion", "tejasbargujepatil/onion-diseases"),
    ("Rose", "shuvokumarbasak4004/rose-leaf-disease-dataset"),
    ("Corn_Maize", "smaranjitghose/corn-or-maize-leaf-disease-dataset"),
    ("Soybean", "sivm205/soybean-diseased-leaf-dataset"),
    ("Chickpea", "tejus0017/chickpea-diseases"),
    ("Apple", "mfaizankb/apple-disease-dataset"),
    ("Pepper", "alexanderuzhinskiy/pepper-disease-classification-dataset-doctorp"),
    ("Sugarcane", "nirmalgaud/sugarcane-leaf-disease-dataset"),
    ("Cotton", "nguynphancminh/cotton-leaf-diseases-dataset"),
    ("Indian_Mustard", "bintangpratomo/mustard-leaf-dataset"),
]

downloaded_dirs = []
print("\n2. Fetching datasets into cloud memory...")
for crop, slug in DATASET_SLUGS:
    try:
        p = kagglehub.dataset_download(slug)
        print(f"   [OK] {crop:15s} -> {slug}")
        downloaded_dirs.append((crop, Path(p)))
    except Exception as e:
        print(f"   [Notice] {crop:15s} failed: {e}")

# Also check /kaggle/input in case user attached any manually
if Path("/kaggle/input").exists():
    for p in Path("/kaggle/input").iterdir():
        if p.is_dir():
            downloaded_dirs.append(("Auto", p))

# ------------------------------------------------------------------------------
# 2. UNIFY ALL DATASETS INTO /kaggle/working/unified_dataset
# ------------------------------------------------------------------------------
BASE_UNIFIED = Path("/kaggle/working/unified_dataset")
if BASE_UNIFIED.exists():
    shutil.rmtree(BASE_UNIFIED)
BASE_UNIFIED.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
total_images = 0
classes_found = set()

print("\n3. Organizing & indexing classes...")
for crop, root_path in downloaded_dirs:
    for img_file in root_path.rglob("*"):
        if img_file.is_file() and img_file.suffix.lower() in IMAGE_EXTS:
            parent_folder = img_file.parent.name
            clean_name = re.sub(r'[^a-zA-Z0-9_]', '_', parent_folder).strip('_')
            
            # Avoid generic names like 'train', 'test', 'images'
            if clean_name.lower() in {'train', 'test', 'val', 'valid', 'validation', 'images', 'data'}:
                clean_name = re.sub(r'[^a-zA-Z0-9_]', '_', img_file.parent.parent.name).strip('_')
            
            if "___" in clean_name:
                class_label = clean_name
            else:
                class_label = f"{crop}___{clean_name}"
            
            target_class_dir = BASE_UNIFIED / class_label
            target_class_dir.mkdir(exist_ok=True)
            classes_found.add(class_label)
            
            dest = target_class_dir / f"{total_images}_{img_file.name}"
            try:
                os.symlink(img_file, dest)
            except Exception:
                shutil.copy2(img_file, dest)
            total_images += 1

print(f"   [SUCCESS] Indexed {total_images:,} images across {len(classes_found)} classes!")

if total_images == 0:
    raise RuntimeError("No images could be gathered. Please check your internet connection!")

# Save classes.json
sorted_classes = sorted(list(classes_found))
classes_dict = {str(i): c for i, c in enumerate(sorted_classes)}
with open("/kaggle/working/classes.json", "w", encoding="utf-8") as f:
    json.dump(classes_dict, f, indent=2)
print("   [OK] Saved /kaggle/working/classes.json")

# ------------------------------------------------------------------------------
# 3. BUILD KERAS DATA LOADERS
# ------------------------------------------------------------------------------
print("\n4. Initializing Keras DataLoaders...")
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = keras.utils.image_dataset_from_directory(
    BASE_UNIFIED,
    validation_split=0.2,
    subset='training',
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    follow_links=True
)

val_ds = keras.utils.image_dataset_from_directory(
    BASE_UNIFIED,
    validation_split=0.2,
    subset='validation',
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    follow_links=True
)

NUM_CLASSES = len(train_ds.class_names)
print(f"   [OK] Detected {NUM_CLASSES} classes in DataLoader!")

# Data Augmentation on GPU
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15)
])

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

# ------------------------------------------------------------------------------
# 4. BUILD MOBILENETV2 MODEL
# ------------------------------------------------------------------------------
print("\n5. Building MobileNetV2 architecture...")
inputs = keras.Input(shape=(3, 224, 224), name="input")
x = layers.Permute((2, 3, 1))(inputs)  # NCHW -> NHWC for Keras backbone

base_model = keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, name="output")(x)

model = keras.Model(inputs=inputs, outputs=outputs, name="crop_disease_model")
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

# ------------------------------------------------------------------------------
# 5. TRAIN MODEL (FAST 6 EPOCHS PHASE 1 + 4 EPOCHS FINE-TUNING)
# ------------------------------------------------------------------------------
print("\n6. Training model on GPU...")
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=6,
    callbacks=callbacks
)

# Phase 2 Fine-tuning
print("\n7. Fine-tuning top 30 layers...")
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=4,
    callbacks=callbacks
)

# ------------------------------------------------------------------------------
# 6. EXPORT TO crop_model.onnx
# ------------------------------------------------------------------------------
print("\n8. Exporting to crop_model.onnx...")
output_onnx_path = "/kaggle/working/crop_model.onnx"
input_signature = [tf.TensorSpec([None, 3, 224, 224], tf.float32, name="input")]

onnx_model, _ = tf2onnx.convert.from_keras(
    model,
    input_signature=input_signature,
    opset=14,
    output_path=output_onnx_path
)

# Verify with ONNX Runtime
session = ort.InferenceSession(output_onnx_path, providers=["CPUExecutionProvider"])
dummy = np.random.randn(1, 3, 224, 224).astype(np.float32)
out = session.run(None, {"input": dummy})
print("   [OK] Inference verified! Output shape:", out[0].shape)

print("\n" + "=" * 80)
print("🎉 ALL DONE! Training & Export Complete!")
print(f"1. Model saved:   {output_onnx_path}")
print(f"2. Classes saved: /kaggle/working/classes.json")
print("=" * 80)
print("👉 Now simply download 'crop_model.onnx' from the right-side Output tab!")

